# Code to find potential matches for each company by scraping search results for a given company name

### Code to scrape possible profile matches of a single company (Atmen)

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
import time

# Set up the Chrome WebDriver with the new method
service = Service(ChromeDriverManager().install())
options = webdriver.ChromeOptions()

# Instantiate the driver with the service argument
driver = webdriver.Chrome(service=service, options=options)

# Open the webpage
driver.get('https://pitchbook.com/profiles/search?q=atmen')

# Wait for the page to load completely
time.sleep(5)  # Adding wait to ensure the page is loaded

# Locate the list of profiles
profile_list = driver.find_elements(By.CSS_SELECTOR, 'ul.profile-list.list-type-none li a')

# Extract profile names and URLs
profiles = []
for profile in profile_list:
    name = profile.get_attribute('title')
    url = profile.get_attribute('href')
    profiles.append({'name': name, 'url': url})

# Print the extracted profiles
for profile in profiles:
    print(f"Name: {profile['name']}, URL: {profile['url']}")

# Close the browser window after extraction
driver.quit()

Name: Atmen, URL: https://pitchbook.com/profiles/company/461834-47
Name: Atmen Pharmaceutical, URL: https://pitchbook.com/profiles/company/664346-89
Name: Atmen Biomedical, URL: https://pitchbook.com/profiles/company/339879-70
Name: Atmen (Environmental Services), URL: https://pitchbook.com/profiles/company/517942-63


### Code to load the dataset provided by MV and pre-process it 

In [ ]:
import pandas as pd


/Users/nipunbhatia/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [ ]:
df = pd.read_csv("/Users/nipunbhatia/Desktop/MV/MV_Dealflow_Funnel_NewLeads_Test_Nipun - Sheet1.csv")

In [ ]:
df.head(25)

,Organization Id,Name,Website,Status
0,1,Cloover,cloover.co,New Leads
1,2,QuID Cash,quidcash.in,New Leads
2,3,ICODOS,icodos.com,New Leads
3,4,Spydra/blog,spydra.app,New Leads
4,5,Cargado,cargado.com,New Leads
5,6,CNaught,cnaught.com,New Leads
6,7,Monterra (formerly CarbEngage),monterra.ai,New Leads
7,8,Metris Energy,metrisenergy.com,New Leads
8,9,Atmen,atmen.co,New Leads
9,10,Bull Agritech,bullagritech.in,New Leads


In [ ]:
def process_company_name(name):
    # Split the name by spaces
    name_parts = name.split()

    # Keep only the first two words
    if len(name_parts) > 2:
        name_parts = name_parts[:2]

    # Join the words back with a space and replace spaces with "+"
    processed_name = " ".join(name_parts).replace(" ", "+")

    return processed_name


In [ ]:
# Apply the processing function to the "Name" column
df['Processed Name'] = df['Name'].apply(process_company_name)

# Now you can use 'Processed Name' for scraping


In [ ]:
df.head(30)

,Organization Id,Name,Website,Status,Processed Name
0,1,Cloover,cloover.co,New Leads,Cloover
1,2,QuID Cash,quidcash.in,New Leads,QuID+Cash
2,3,ICODOS,icodos.com,New Leads,ICODOS
3,4,Spydra/blog,spydra.app,New Leads,Spydra/blog
4,5,Cargado,cargado.com,New Leads,Cargado
5,6,CNaught,cnaught.com,New Leads,CNaught
6,7,Monterra (formerly CarbEngage),monterra.ai,New Leads,Monterra+(formerly
7,8,Metris Energy,metrisenergy.com,New Leads,Metris+Energy
8,9,Atmen,atmen.co,New Leads,Atmen
9,10,Bull Agritech,bullagritech.in,New Leads,Bull+Agritech


### Code to scrape the possible matvhes of every company present in the csv

In [3]:
import random
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

# Function to set up Selenium WebDriver with safety mechanisms
def get_driver():
    options = Options()
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--disable-infobars")
    options.add_argument("--start-maximized")
    options.add_argument("--incognito")
    options.add_experimental_option("excludeSwitches", ["enable-automation"])
    options.add_experimental_option("useAutomationExtension", False)
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/89.0.4389.82 Safari/537.36"
    )

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    
    # Mask WebDriver properties
    driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
    driver.execute_script("""
        const newProto = navigator.__proto__;
        delete newProto.webdriver;
        navigator.__proto__ = newProto;
    """)
    return driver

# Introduce random delays
def random_delay():
    delay = random.uniform(5, 12)
    time.sleep(delay)

In [4]:


# Scrape PitchBook companies
def scrape_pitchbook_companies(driver, search_query):
    url = f"https://pitchbook.com/profiles/search?q={search_query}"
    try:
        print(f"Accessing {url}")
        driver.get(url)
        
        # Allow the page to load
        random_delay()
        
        # Locate the list of profiles
        profile_elements = driver.find_elements(By.CSS_SELECTOR, 'ul.profile-list.list-type-none li')
        
        # Filter and extract companies only
        companies = []
        for profile in profile_elements:
            try:
                category = profile.find_element(By.CSS_SELECTOR, "span.text-small").text
                if category == "Company":  # Only take "Company" entries
                    name = profile.find_element(By.CSS_SELECTOR, "a").get_attribute("title")
                    link = profile.find_element(By.CSS_SELECTOR, "a").get_attribute("href")
                    companies.append({"name": name, "link": link})
            except Exception as e:
                print(f"Error processing a profile: {e}")
        
        return companies

    except Exception as e:
        print(f"Error scraping search query '{search_query}': {e}")
        return []

# Main function to scrape multiple search queries
def scrape_multiple_searches(search_queries):
    results = []
    driver = get_driver()
    
    try:
        for query in search_queries:
            print(f"Processing query: {query}")
            companies = scrape_pitchbook_companies(driver, query)
            results.extend(companies)
            print("Adding random delay after processing query.")
            random_delay()
    
    finally:
        driver.quit()
    
    return results


In [ ]:
import pandas as pd

# Ensure your DataFrame `df` is already loaded with columns "Name" and "Website"
# For example, df = pd.read_excel("your_file.xlsx")

def scrape_companies_from_df(df):
    # Initialize an empty list to store results
    all_results = []

    # Create and initialize Selenium WebDriver
    driver = get_driver()

    try:
        # Iterate over each company name in the DataFrame
        for index, row in df.iterrows():
            search_query = row['Processed Name']
            print(f"Scraping for company: {search_query}")
            
            # Use the scraping function to extract companies
            results = scrape_pitchbook_companies(driver, search_query)
            
            # Add a "Query" column to track the original company name
            for result in results:
                result['Query'] = search_query
            
            # Append results to the all_results list
            all_results.extend(results)
            print(results)
            # Random delay between queries
            random_delay()

    finally:
        # Quit the driver after scraping
        driver.quit()

    # Convert all_results to a DataFrame
    
    results_df = pd.DataFrame(all_results)
    return results_df

# Scrape and save results
if __name__ == "__main__":
    # Ensure your DataFrame `df` is already loaded
    extracted_data = scrape_companies_from_df(df)
    
    


Scraping for company: Cloover
Accessing https://pitchbook.com/profiles/search?q=Cloover
[{'name': 'Cloover', 'link': 'https://pitchbook.com/profiles/company/518899-51', 'Query': 'Cloover'}, {'name': 'iDaily', 'link': 'https://pitchbook.com/profiles/company/99371-62', 'Query': 'Cloover'}]
Scraping for company: QuID+Cash
Accessing https://pitchbook.com/profiles/search?q=QuID+Cash
[{'name': 'Mobicash America', 'link': 'https://pitchbook.com/profiles/company/94877-56', 'Query': 'QuID+Cash'}]
Scraping for company: ICODOS
Accessing https://pitchbook.com/profiles/search?q=ICODOS
[{'name': 'ICODOS', 'link': 'https://pitchbook.com/profiles/company/533276-02', 'Query': 'ICODOS'}]
Scraping for company: Spydra/blog
Accessing https://pitchbook.com/profiles/search?q=Spydra/blog
[]
Scraping for company: Cargado
Accessing https://pitchbook.com/profiles/search?q=Cargado
[{'name': 'Cargado', 'link': 'https://pitchbook.com/profiles/company/553176-46', 'Query': 'Cargado'}]
Scraping for company: CNaught
Ac

In [ ]:
extracted_data.head(20)

,Query,name,link
0,Cloover,Cloover,https://pitchbook.com/profiles/company/518899-51
1,Cloover,iDaily,https://pitchbook.com/profiles/company/99371-62
2,QuID+Cash,Mobicash America,https://pitchbook.com/profiles/company/94877-56
3,ICODOS,ICODOS,https://pitchbook.com/profiles/company/533276-02
4,Cargado,Cargado,https://pitchbook.com/profiles/company/553176-46
5,CNaught,CNaught,https://pitchbook.com/profiles/company/510488-65
6,Metris+Energy,Metris Energy,https://pitchbook.com/profiles/company/541282-24
7,Atmen,Atmen,https://pitchbook.com/profiles/company/461834-47
8,Atmen,Atmen Pharmaceutical,https://pitchbook.com/profiles/company/664346-89
9,Atmen,Atmen Biomedical,https://pitchbook.com/profiles/company/339879-70


In [ ]:
extracted_data.to_csv("Pitchbook_companies.csv")